# 爬證交所資料

In [2]:
import pandas as pd
import numpy as np
import json
import requests
import time
import os

# folder_list = os.listdir(os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news")
# folder_list = [folder_list for folder_list in folder_list if folder_list != '.DS_Store']
# print(folder_list)

# 建立一個字典，其中包含所有欄位名稱的中文到英文的映射
column_name_mapping = {
    '日期': 'Date',
    '成交股數': 'Volume',
    '成交金額': 'Transaction Amount',
    '開盤價': 'Open',
    '最高價': 'High',
    '最低價': 'Low',
    '收盤價': 'Close',
    '漲跌價差': 'Diff',
    '成交筆數': 'Transaction Count'
}

def fetch_data_twse(ticker_symbol, year1, year2):
    print("Fetching data for " + ticker_symbol + " from " + str(year1) + " to " + str(year2) + "...")
    df_y = pd.DataFrame()
    for i in range(year1, year2+1):
        for j in range(12):
            month = str(i) + str(j+1).zfill(2)
            url = f'https://www.twse.com.tw/exchangeReport/STOCK_DAY?response=json&date={month}01&stockNo={ticker_symbol}'
            print(url)
            data = requests.get(url).text
            data = json.loads(data)
            if data['stat'] != 'OK':
                print('Data not found for ' + month)
                break
            df_m = pd.DataFrame(data['data']) 
            df_m.columns = data['fields']
            df_m = df_m.rename(columns=column_name_mapping)
            df_m.set_index('Date', inplace=True)
            df_y = pd.concat([df_y, df_m])
            print(df_y)
            # time.sleep(1)

    # 將日期從民國年轉換為西元年
    df_y.index = pd.Series(df_y.index).str.replace('/', '').apply(lambda x: str(int(x[:3]) + 1911) + x[3:])

    return df_y


In [13]:
path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw', 'stock_price')
folder_list = os.listdir(os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news")
folder_list = [folder_list for folder_list in folder_list if folder_list != '.DS_Store']
print(folder_list)
folder_list = ['2330']

for id in folder_list:
    try:
        stock_tech = fetch_data_twse(id, 2022, 2024) # 這裡可以自行設定要抓取的時間範圍
        stock_tech.to_csv(os.path.join(path , id + 'twse.csv')) # 將資料存成 csv 檔
        print('save ' + id + 'twse.csv')
    except Exception as e:
        print("Error: " + id + " " + str(e))
        pass

['2618', '2881', '2880', '6281', '8111', '2211', '2731', '2707', '3023', '9941', '2308', '6104', '1514', '3687', '1101', '2542', '5904', '3036', '2385', '1303', '2912', '2323', '2603', '1419', '3537', '1604', '2486', '5269', '1314', '8033', '2704', '3017', '2903', '9943', '2368', '9904', '1102', '3231', '3004', '3035', '3264', '6176', '8027', '1336', '6113', '2311', '1103', '4736', '2609', '2002']
Fetching data for 2330 from 2022 to 2024...
https://www.twse.com.tw/exchangeReport/STOCK_DAY?response=json&date=20220101&stockNo=2330
               Volume Transaction Amount    Open    High     Low   Close  \
Date                                                                       
111/01/03  73,703,302     46,249,716,919  619.00  632.00  618.00  631.00   
111/01/04  90,945,643     59,188,199,534  645.00  656.00  644.00  656.00   
111/01/05  72,505,550     47,582,832,784  669.00  669.00  646.00  650.00   
111/01/06  57,490,736     36,817,638,522  638.00  646.00  636.00  644.00   
111/01/07

# 畫技術分析


In [16]:
path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw', 'stock_price')

id = '2330'
stock_tech = pd.read_csv(os.path.join(path , id + 'twse.csv'))

# 將stock_tech的值都換成float
stock_tech['Open'] = stock_tech['Open'].map(lambda x: float(x.replace(',', '')))
stock_tech['Close'] = stock_tech['Close'].map(lambda x: float(x.replace(',', '')))
stock_tech['High'] = stock_tech['High'].map(lambda x: float(x.replace(',', '')))
stock_tech['Low'] = stock_tech['Low'].map(lambda x: float(x.replace(',', '')))


# 計算移動均線 (SMA)
stock_tech['MA5'] = stock_tech['Close'].rolling(window=5).mean()
stock_tech['MA10'] = stock_tech['Close'].rolling(window=10).mean()
stock_tech['MA20'] = stock_tech['Close'].rolling(window=20).mean()

# 計算指數移動平均線 (EMA)
stock_tech['EMA12'] = stock_tech['Close'].ewm(span=12, adjust=False).mean()
stock_tech['EMA26'] = stock_tech['Close'].ewm(span=26, adjust=False).mean()

# 計算MACD和DIF
stock_tech['DIF'] = stock_tech['EMA12'] - stock_tech['EMA26']
stock_tech['MACD'] = stock_tech['DIF'].ewm(span=9, adjust=False).mean()


# 計算RSI
def calculate_rsi(series, period=14):
    delta = series.diff(1)
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

stock_tech['RSI'] = calculate_rsi(stock_tech['Close'])

# 計算KDJ
low_list = stock_tech['Low'].rolling(9, min_periods=9).min()
low_list.fillna(value=stock_tech['Low'].expanding().min(), inplace=True)
high_list = stock_tech['High'].rolling(9, min_periods=9).max()
high_list.fillna(value=stock_tech['High'].expanding().max(), inplace=True)
rsv = (stock_tech['Close'] - low_list) / (high_list - low_list) * 100


# 初始值設定
stock_tech['K'] = 50
stock_tech['D'] = 50

for i in range(1, len(stock_tech)):
    stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
    stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']


# 顯示DataFrame
stock_tech.to_csv(os.path.join(path , id + 'twse.csv'), encoding='utf-8', index=False)

/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_72889/2351713471.py:51: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '66.66666666666666' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_72889/2351713471.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '55.55555555555554' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']
